# MAX-2 · Ordinal Training Library (5-class CORN) — Zero-Touch

Defines the maximum-configuration training stack for five-class Kellgren–Lawrence
grading. All maximum-stack parameters and output paths are defined inside this
library: it reads only the shared data paths (manifest, image array) from the
original configuration and writes every checkpoint and result to a separate
folder (scope3_max). The original config.py, training_lib.py, and all prior
results are never modified.

The classification head is a rank-consistent ordinal head (CORN): five ordered
grades are predicted as four cumulative thresholds P(grade > k). The stack also
provides soft ordinal targets, source-reliability weighting, full-backbone
fine-tuning, and a configurable input resolution.

## Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import sys, importlib
sys.path.insert(0,'/content/drive/MyDrive/Master Thesis/scope3')
import config; importlib.reload(config)
print('Original config loaded (read-only). Shared data:', config.MANIFEST_CSV)

Mounted at /content/drive
Original config loaded (read-only). Shared data: /content/drive/MyDrive/Master Thesis/scope3/manifest.csv


## Write self-contained ordinal library

All maximum-stack settings (resolution, freeze level, learning rate, batch size,
MRKR trust, output folders) are defined within the library itself.

In [2]:
import sys
sys.path.insert(0,'/content/drive/MyDrive/Master Thesis/scope3')
lib_code = r'''# MAX ordinal library (zero-touch) — defines its own params + output paths; reads only shared data from config
import os, json, time, math, random
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.autograd import Function
from pathlib import Path
from PIL import Image
import config   # original config: used ONLY for shared data paths + a few constants

# ===== MAX stack parameters (defined here, not in config.py) =====
MAX_IMG_SIZE      = 384        # high-resolution training
MAX_FREEZE_STAGES = 0          # 0 = full backbone fine-tune
MAX_LR_BACKBONE   = 5e-6       # low LR for stable full fine-tune
MAX_LR_HEAD       = 1e-4
MAX_BATCH_SIZE    = 12         # fits 384 in A100 memory
MAX_GRAD_ACCUM    = 4          # effective batch ~48
MAX_SOFT_ALPHA    = 0.15       # soft adjacent smoothing on thresholds
MRKR_TRUST        = 0.4        # source-reliability weight for MRKR machine labels
MAX_EPOCHS        = getattr(config,"EPOCHS",18)
MAX_WARMUP        = getattr(config,"WARMUP_EPOCHS",2)
MAX_PATIENCE      = getattr(config,"EARLY_STOP_PATIENCE",6)

# ===== Separate output folders (scope3_max) — originals untouched =====
PROJECT_ROOT = Path("/content/drive/MyDrive/Master Thesis")
MAX_ROOT     = PROJECT_ROOT/"scope3_max"
MAX_CKPT_DIR = MAX_ROOT/"checkpoints"
MAX_RESULTS_DIR = MAX_ROOT/"results"
for d in (MAX_ROOT, MAX_CKPT_DIR, MAX_RESULTS_DIR): d.mkdir(parents=True, exist_ok=True)

# ===== Shared data + constants from original config (read-only) =====
MANIFEST_CSV = config.MANIFEST_CSV
IMAGES_NPY   = config.IMAGES_NPY
DATASETS     = config.DATASETS
LODO_FOLDS   = config.LODO_FOLDS
SEEDS        = config.SEEDS
K            = config.NUM_CLASSES
BASE_IMG     = config.IMG_SIZE
LOSS_WEIGHTS = config.LOSS_WEIGHTS
CURRICULUM   = config.CURRICULUM
STAGE1_OA    = config.STAGE1_OA_THRESHOLD
TTA_TRANSFORMS = getattr(config,"TTA_TRANSFORMS",["identity","hflip"])
USE_TTA      = getattr(config,"USE_TTA",True)
USE_JOINT_CROP = getattr(config,"USE_JOINT_CROP",True)
JOINT_CROP_FRAC = getattr(config,"JOINT_CROP_FRAC",0.80)
WEIGHT_DECAY = getattr(config,"WEIGHT_DECAY",1e-4)
QUALITY_MIN_WEIGHT = getattr(config,"QUALITY_MIN_WEIGHT",0.2)
LABEL_QUALITY_CSV  = getattr(config,"LABEL_QUALITY_CSV", MAX_RESULTS_DIR/"none.csv")
NUM_WORKERS  = getattr(config,"NUM_WORKERS",2)
GRL_LAMBDA_MAX = getattr(config,"GRL_LAMBDA_MAX",0.3)
DATASET_TO_IDX = {"oai":0,"nhanes3":1,"mrkr":2,"mendeley":3}
SAVE_PREDICTIONS = True
_IMAGES = None

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def joint_crop(arr):
    if not USE_JOINT_CROP: return arr
    h,w=arr.shape[:2]; f=JOINT_CROP_FRAC; ch,cw=int(h*f),int(w*f); top,left=(h-ch)//2,(w-cw)//2
    return arr[top:top+ch, left:left+cw]

def prepare_local_data(force=False):
    global _IMAGES
    import shutil
    man=pd.read_csv(str(MANIFEST_CSV)); local="/content/images.npy"
    if _IMAGES is not None and not force and len(_IMAGES)>=len(man)*0.99:
        print(f"Array already loaded ({len(_IMAGES):,}).");
        if "arr_idx" not in man.columns: man["arr_idx"]=np.arange(len(man))
        return man
    if os.path.exists(str(IMAGES_NPY)):
        if not os.path.exists(local):
            t0=time.time(); shutil.copy(str(IMAGES_NPY),local); print(f"Copied array in {time.time()-t0:.0f}s")
        t1=time.time(); _IMAGES=np.load(local); print(f"Loaded array {tuple(_IMAGES.shape)} in {time.time()-t1:.0f}s")
    if "arr_idx" not in man.columns: man["arr_idx"]=np.arange(len(man))
    return man

def _get_image(row):
    global _IMAGES
    if _IMAGES is not None and "arr_idx" in row: return _IMAGES[int(row["arr_idx"])]
    return np.array(Image.open(row["abs_path"]).convert("L"),dtype=np.uint8)

def _resize(a):
    import cv2
    if a.shape!=(MAX_IMG_SIZE,MAX_IMG_SIZE): a=cv2.resize(a,(MAX_IMG_SIZE,MAX_IMG_SIZE),interpolation=cv2.INTER_AREA)
    return a

def load_quality_weights():
    q={}
    try:
        df=pd.read_csv(str(LABEL_QUALITY_CSV))
        for _,r in df.iterrows(): q[r["filename"]]=max(QUALITY_MIN_WEIGHT,min(1.0,QUALITY_MIN_WEIGHT+(1.0-QUALITY_MIN_WEIGHT)*float(r["label_quality"])))
    except Exception: pass
    return q

class KneeDataset(Dataset):
    def __init__(self, df, train=True, quality=None):
        self.df=df.reset_index(drop=True); self.train=train; self.quality=quality or {}
    def __len__(self): return len(self.df)
    def _aug(self,a):
        if not self.train: return a
        import cv2
        if random.random()<0.5: a=np.fliplr(a).copy()
        if random.random()<0.5:
            ang=random.uniform(-7,7); h,w=a.shape; M=cv2.getRotationMatrix2D((w/2,h/2),ang,1.0); a=cv2.warpAffine(a,M,(w,h),borderMode=cv2.BORDER_REFLECT)
        if random.random()<0.3:
            f=random.uniform(0.9,1.1); a=np.clip(a.astype(np.float32)*f,0,255).astype(np.uint8)
        return a
    def __getitem__(self,i):
        r=self.df.iloc[i]; a=self._aug(_resize(joint_crop(_get_image(r))))
        x=torch.from_numpy(a.astype(np.float32)/255.0); x=(x-0.485)/0.229; x=x.unsqueeze(0).repeat(3,1,1)
        y=int(r["kl_grade"]); base=LOSS_WEIGHTS.get(r["kl_source"],1.0)
        # source-reliability: MRKR machine labels down-weighted by MRKR_TRUST
        if r["dataset"]=="mrkr": base=base*MRKR_TRUST
        qw=self.quality.get(r["filename"],1.0) if r["kl_source"]=="model_predicted" else 1.0
        return x,y,torch.tensor(base*qw,dtype=torch.float32),DATASET_TO_IDX.get(r["dataset"],0)

def build_sampler(df, mrkr_weight=1.0, clean_only=False):
    counts=df["dataset"].value_counts().to_dict(); w=np.zeros(len(df)); src=df["kl_source"].values; ds=df["dataset"].values
    for i in range(len(df)):
        b=1.0/counts[ds[i]]
        if src[i]=="model_predicted": b=0.0 if clean_only else b*mrkr_weight
        w[i]=b
    if w.sum()==0: w[:]=1.0
    return WeightedRandomSampler(torch.as_tensor(w,dtype=torch.double),len(df),replacement=True)

class GradReverse(Function):
    @staticmethod
    def forward(ctx,x,l): ctx.l=l; return x.view_as(x)
    @staticmethod
    def backward(ctx,g): return g.neg()*ctx.l, None
def grad_reverse(x,l=1.0): return GradReverse.apply(x,l)

class OrdinalNet(nn.Module):
    def __init__(self, num_classes=5, n_domains=4, use_hierarchical=True):
        super().__init__(); self.K=num_classes; self.use_hierarchical=use_hierarchical
        from torchvision import models
        try:
            from torchvision.models import ConvNeXt_Large_Weights
            self.backbone=models.convnext_large(weights=ConvNeXt_Large_Weights.IMAGENET1K_V1)
        except Exception: self.backbone=models.convnext_large(weights="IMAGENET1K_V1")
        feat=self.backbone.classifier[2].in_features; self.backbone.classifier=nn.Identity(); self.feat_dim=feat
        self.corn=nn.Sequential(nn.Flatten(1),nn.LayerNorm(feat),nn.Dropout(0.3),nn.Linear(feat,num_classes-1))
        self.head_s1=nn.Sequential(nn.Flatten(1),nn.Dropout(0.3),nn.Linear(feat,2))
        self.head_s2=nn.Sequential(nn.Flatten(1),nn.Dropout(0.3),nn.Linear(feat,2))
        self.domain_head=nn.Sequential(nn.Flatten(1),nn.Dropout(0.3),nn.Linear(feat,128),nn.ReLU(),nn.Linear(128,n_domains))
    def forward(self,x,grl_lambda=0.0):
        f=self.backbone(x)
        if f.dim()==4: f=f.mean(dim=[-2,-1])
        return self.corn(f), self.head_s1(f), self.head_s2(f), self.domain_head(grad_reverse(f,grl_lambda))
    def freeze_stages(self,n):
        for idx,block in enumerate(self.backbone.features):
            req=idx>=(n*2)
            for p in block.parameters(): p.requires_grad=req
    def param_groups(self):
        hp,bp=[],[]
        for n,p in self.named_parameters():
            if p.requires_grad: (hp if ("corn" in n or "head" in n) else bp).append(p)
        return hp,bp

def corn_targets(y,K):
    lv=torch.arange(K-1,device=y.device).unsqueeze(0)
    return (y.unsqueeze(1)>lv).float(), (y.unsqueeze(1)>=lv).float()
def corn_loss(logits,y,lw,soft_alpha=0.0):
    K=logits.shape[1]+1; tgt,mask=corn_targets(y,K)
    if soft_alpha>0: tgt=tgt*(1.0-soft_alpha)+0.5*soft_alpha
    bce=F.binary_cross_entropy_with_logits(logits,tgt,reduction="none")
    bce=(bce*mask).sum(1)/mask.sum(1).clamp(min=1)
    return (bce*lw).mean()
def corn_predict(logits): return (torch.sigmoid(logits)>0.5).sum(1)
def corn_probs(logits):
    p=torch.sigmoid(logits); B,Km1=p.shape
    cum=torch.cat([torch.ones(B,1,device=p.device),p,torch.zeros(B,1,device=p.device)],dim=1)
    cls=(cum[:,:-1]-cum[:,1:]).clamp(min=1e-6); return cls/cls.sum(1,keepdim=True)
def hier_aux_loss(s1,s2,y):
    t1=(y>=STAGE1_OA).long(); c1=F.cross_entropy(s1,t1); oa=t1.bool(); c2=torch.tensor(0.0,device=y.device)
    if oa.any(): c2=F.cross_entropy(s2[oa],(y[oa]==4).long())
    return 0.5*c1+0.5*c2
def domain_loss(d,t): return F.cross_entropy(d,t)

def save_ckpt(p,m,o,e,bv,h):
    t=str(p)+".tmp"; torch.save({"model":m.state_dict(),"opt":o.state_dict(),"epoch":e,"best_val":bv,"history":h},t); os.replace(t,str(p))
def load_ckpt(p,m,o):
    ck=torch.load(str(p),map_location="cpu"); m.load_state_dict(ck["model"])
    if o is not None and "opt" in ck: o.load_state_dict(ck["opt"])
    return ck["epoch"],ck.get("best_val",0.0),ck.get("history",[])

@torch.no_grad()
def predict_tta(model,df,device,batch_size=12,use_tta=True):
    import cv2; model.eval(); df=df.reset_index(drop=True); preds=[]; probs=[]
    def ln(a):
        x=torch.from_numpy(a.astype(np.float32)/255.0); x=(x-0.485)/0.229; return x.unsqueeze(0).repeat(3,1,1)
    tfs=TTA_TRANSFORMS if use_tta else ["identity"]
    for s in range(0,len(df),batch_size):
        sub=df.iloc[s:s+batch_size]; acc=None
        for tn in tfs:
            b=[]
            for _,r in sub.iterrows():
                a=_resize(joint_crop(_get_image(r)))
                if tn=="hflip": a=np.fliplr(a).copy()
                elif tn=="rot+5":
                    h,w=a.shape; M=cv2.getRotationMatrix2D((w/2,h/2),5,1.0); a=cv2.warpAffine(a,M,(w,h),borderMode=cv2.BORDER_REFLECT)
                elif tn=="rot-5":
                    h,w=a.shape; M=cv2.getRotationMatrix2D((w/2,h/2),-5,1.0); a=cv2.warpAffine(a,M,(w,h),borderMode=cv2.BORDER_REFLECT)
                b.append(ln(a))
            xb=torch.stack(b).to(device); lo,_,_,_=model(xb,grl_lambda=0.0)
            p=corn_probs(lo).cpu().numpy(); acc=p if acc is None else acc+p
        acc/=len(tfs); probs.append(acc); preds.extend(acc.argmax(1).tolist())
    return np.array(preds), np.vstack(probs)

def compute_metrics(yt,yp,probs=None):
    from sklearn.metrics import accuracy_score,cohen_kappa_score,roc_auc_score,f1_score
    yt=np.asarray(yt); yp=np.asarray(yp); m={}; d=np.abs(yt-yp)
    m["acc5"]=float(accuracy_score(yt,yp)); m["qwk"]=float(cohen_kappa_score(yt,yp,weights="quadratic")); m["within1"]=float((d<=1).mean())
    bt=(yt>=2).astype(int); bp=(yp>=2).astype(int); m["acc_binary"]=float(accuracy_score(bt,bp))
    mk=np.isin(yt,[1,2]); m["f1_kl12"]=float(f1_score(yt[mk],np.clip(yp[mk],1,2),labels=[1,2],average="macro")) if mk.sum()>0 else float("nan")
    if probs is not None:
        try: m["auc_oa"]=float(roc_auc_score(bt,probs[:,2:].sum(1)))
        except Exception: m["auc_oa"]=float("nan")
        oa=yt>=2
        if oa.sum()>10 and len(np.unique((yt[oa]==4)))>1:
            try: m["auc_severity"]=float(roc_auc_score((yt[oa]==4).astype(int),probs[oa,4]/(probs[oa,2:].sum(1)+1e-8)))
            except Exception: m["auc_severity"]=float("nan")
        else: m["auc_severity"]=float("nan")
    return m

def run_training(run_name,train_df,val_df,test_df,mechanisms,seed,ckpt_dir=None,results_dir=None,log_fn=print):
    set_seed(seed)
    ckpt_dir=ckpt_dir or MAX_CKPT_DIR; results_dir=results_dir or MAX_RESULTS_DIR
    if not torch.cuda.is_available(): raise RuntimeError("No GPU.")
    device="cuda"; done=os.path.join(str(results_dir),run_name+".json")
    if os.path.exists(done):
        with open(done) as f: log_fn(f"  [{run_name}] complete - skip."); return json.load(f)
    use_hier=mechanisms.get("hierarchical",True); use_dom=mechanisms.get("domain_adv",False); use_q=mechanisms.get("use_quality",False)
    soft_alpha=mechanisms.get("soft_alpha",MAX_SOFT_ALPHA); freeze=mechanisms.get("freeze_stages",MAX_FREEZE_STAGES)
    bs=mechanisms.get("batch_size",MAX_BATCH_SIZE); accum=mechanisms.get("grad_accum",MAX_GRAD_ACCUM); lr_bb=mechanisms.get("lr_backbone",MAX_LR_BACKBONE)
    quality=load_quality_weights() if use_q else {}
    model=OrdinalNet(K,4,use_hier).to(device); model.freeze_stages(freeze); hp,bp=model.param_groups()
    opt=torch.optim.AdamW([{"params":hp,"lr":MAX_LR_HEAD},{"params":bp,"lr":lr_bb}],weight_decay=WEIGHT_DECAY)
    scaler=torch.amp.GradScaler("cuda"); ckpt=os.path.join(str(ckpt_dir),run_name+".pt"); e0,best,hist=0,0.0,[]
    if os.path.exists(ckpt):
        try: e0,best,hist=load_ckpt(ckpt,model,opt); log_fn(f"  [{run_name}] resume ep{e0}")
        except Exception as ex: log_fn(f"  [{run_name}] ckpt corrupt; fresh start"); e0,best,hist=0,0.0,[]
    no_imp=0
    for epoch in range(e0,MAX_EPOCHS):
        if mechanisms.get("curriculum",False):
            if epoch<CURRICULUM["phase1_end"]: clean,mw=True,0.0
            elif epoch<CURRICULUM["phase2_end"]:
                clean=False; fr=(epoch-CURRICULUM["phase1_end"])/max(1,CURRICULUM["phase2_end"]-CURRICULUM["phase1_end"]); mw=fr*CURRICULUM["mrkr_target_weight"]
            else: clean,mw=False,CURRICULUM["mrkr_target_weight"]
        else: clean,mw=False,CURRICULUM["mrkr_target_weight"]
        if mechanisms.get("sampler",False):
            loader=DataLoader(KneeDataset(train_df,True,quality),batch_size=bs,sampler=build_sampler(train_df,mw,clean),num_workers=NUM_WORKERS,pin_memory=True)
        else:
            sub=train_df[train_df["kl_source"]!="model_predicted"] if clean else train_df
            loader=DataLoader(KneeDataset(sub,True,quality),batch_size=bs,shuffle=True,num_workers=NUM_WORKERS,pin_memory=True)
        sc=(epoch+1)/MAX_WARMUP if epoch<MAX_WARMUP else 0.5*(1+math.cos(math.pi*(epoch-MAX_WARMUP)/max(1,MAX_EPOCHS-MAX_WARMUP)))
        opt.param_groups[0]["lr"]=MAX_LR_HEAD*sc; opt.param_groups[1]["lr"]=lr_bb*sc
        lmax=mechanisms.get("grl_lambda_max",GRL_LAMBDA_MAX); grl=(lmax*(2.0/(1.0+math.exp(-10*epoch/max(1,MAX_EPOCHS)))-1.0)) if use_dom else 0.0
        model.train(); t0=time.time(); rl=0.0; nb=0; tc=0; tt=0; opt.zero_grad()
        for bi,(x,y,lw,dom) in enumerate(loader):
            x,y=x.to(device,non_blocking=True),y.to(device,non_blocking=True); lw=lw.to(device); dom=dom.to(device)
            with torch.amp.autocast("cuda"):
                lo,s1,s2,dl=model(x,grl_lambda=grl); loss=corn_loss(lo,y,lw,soft_alpha)
                if use_hier: loss=loss+hier_aux_loss(s1,s2,y)
                if use_dom: loss=loss+domain_loss(dl,dom)
                loss=loss/accum
            scaler.scale(loss).backward()
            if (bi+1)%accum==0:
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); scaler.step(opt); scaler.update(); opt.zero_grad()
            rl+=loss.item()*accum; nb+=1
            with torch.no_grad(): tc+=(corn_predict(lo)==y).sum().item(); tt+=y.size(0)
        tra=tc/max(1,tt)
        vp,vpr=predict_tta(model,val_df,device,bs,use_tta=False); vm=compute_metrics(val_df["kl_grade"].values,vp,vpr); gap=tra-vm["acc5"]
        hist.append({"epoch":epoch,"loss":rl/max(1,nb),"train_acc":tra,"train_val_gap":gap,**vm})
        log_fn(f"  [{run_name}] ep{epoch+1}/{MAX_EPOCHS} loss={rl/max(1,nb):.3f} tr={tra:.3f} val={vm['acc5']:.3f} w1={vm['within1']:.3f} qwk={vm['qwk']:.3f} gap={gap:+.3f} ({time.time()-t0:.0f}s)")
        if vm["acc5"]>best:
            best=vm["acc5"]
            try: save_ckpt(os.path.join(str(ckpt_dir),run_name+"_best.pt"),model,opt,epoch,best,hist)
            except Exception: pass
            no_imp=0
        else: no_imp+=1
        try: save_ckpt(ckpt,model,opt,epoch+1,best,hist)
        except Exception: pass
        if no_imp>=MAX_PATIENCE: log_fn(f"  [{run_name}] early stop ep{epoch+1}"); break
    bpath=os.path.join(str(ckpt_dir),run_name+"_best.pt")
    if os.path.exists(bpath):
        try: load_ckpt(bpath,model,None)
        except Exception: log_fn(f"  [{run_name}] best corrupt; in-memory weights")
    tp,tpr=predict_tta(model,test_df,device,bs,use_tta=USE_TTA); tm=compute_metrics(test_df["kl_grade"].values,tp,tpr); vb=max((h["acc5"] for h in hist),default=0.0)
    if SAVE_PREDICTIONS: np.savez_compressed(os.path.join(str(results_dir),run_name+"_preds.npz"),y_true=test_df["kl_grade"].values,y_pred=tp,probs=tpr)
    res={"run_name":run_name,"seed":seed,"mechanisms":mechanisms,"internal_acc5":vb,"external_acc5":tm["acc5"],
         "external_within1":tm["within1"],"external_qwk":tm["qwk"],"external_acc_binary":tm["acc_binary"],
         "external_auc_oa":tm.get("auc_oa",float("nan")),"external_f1_kl12":tm.get("f1_kl12",float("nan")),
         "gap":vb-tm["acc5"],"n_train":len(train_df),"n_val":len(val_df),"n_test":len(test_df),"history":hist}
    with open(done,"w") as f: json.dump(res,f,indent=2)
    log_fn(f"  [{run_name}] DONE int={vb:.3f} ext={tm['acc5']:.3f} w1={tm['within1']:.3f} qwk={tm['qwk']:.3f} gap={res['gap']:.3f}")
    return res

def make_splits(man, test_ds, seed=0, val_frac=0.15):
    pool=man[man["dataset"]!=test_ds].reset_index(drop=True)
    va=pool.sample(frac=val_frac,random_state=seed); tr=pool.drop(va.index)
    te=man[man["dataset"]==test_ds].reset_index(drop=True)
    return tr.reset_index(drop=True), va.reset_index(drop=True), te
'''
p='/content/drive/MyDrive/Master Thesis/scope3/training_lib_max.py'
with open(p,'w') as f: f.write(lib_code)
print(f'training_lib_max.py written: {len(lib_code):,} bytes')
import importlib
if 'training_lib_max' in sys.modules: importlib.reload(sys.modules['training_lib_max'])
import training_lib_max as TM
print('Zero-touch ordinal library ready.')
print('  Resolution :', TM.MAX_IMG_SIZE)
print('  Freeze     :', TM.MAX_FREEZE_STAGES, '(0 = full fine-tune)')
print('  MRKR trust :', TM.MRKR_TRUST)
print('  Outputs -> ', TM.MAX_RESULTS_DIR)
print('  (original config.py, training_lib.py, scope3/results all untouched)')

training_lib_max.py written: 17,632 bytes
Zero-touch ordinal library ready.
  Resolution : 384
  Freeze     : 0 (0 = full fine-tune)
  MRKR trust : 0.4
  Outputs ->  /content/drive/MyDrive/Master Thesis/scope3_max/results
  (original config.py, training_lib.py, scope3/results all untouched)
